## Install dependencies

In [1]:
!pip install -U kubeflow==0.2.1 --quiet

## Important Next Step: Restart Kernel

The packages have been installed, but to use them in this notebook, **you must restart the kernel**.

### **How to restart:**
1. **Menu bar:** Kernel → Restart Kernel
2. **Shortcut:** `0 0` (press zero twice)
3. **Command palette:** `Ctrl+Shift+P` → "Restart Kernel"


In [4]:
# Verification code to run after restart
import kubeflow
import torch
print(f"Kubeflow version: {kubeflow.__version__ if hasattr(kubeflow, '__version__') else 'N/A'}")
print(f"PyTorch version: {torch.__version__}")
print("All imports successful!")

Kubeflow version: 0.2.1
PyTorch version: 2.3.1+cu121
All imports successful!


In [5]:
import time
from datetime import datetime

import kubeflow
import kubeflow.trainer
from kubeflow.trainer.options import TrainerCommand
from kubeflow.trainer.types.types import CustomTrainerContainer

In [6]:
config = kubeflow.trainer.KubernetesBackendConfig()
trainer = kubeflow.trainer.TrainerClient(backend_config=config)
github_container_registry = (
    "ghcr.io/mxochicale/kubeflowtrainerimage/kubeflowtrainerimage:v0.0.4"
)

# Create environment variables for distributed training
env_vars = {
    "MASTER_ADDR": "localhost",
    "MASTER_PORT": "12355",
    "WORLD_SIZE": "1",
    "RANK": "0",
    "LOCAL_RANK": "0"
}

command = TrainerCommand(command=["./my-entrypoint.sh"])

In [7]:
job_id = trainer.train(
    runtime=trainer.get_runtime("torch-distributed"),
    trainer=CustomTrainerContainer(
        image=github_container_registry,
        env=env_vars        
    ),
    options=[command],
)

In [8]:
#Check job status directly
job = trainer.get_job(job_id)
print(f"\nJob ID: {job_id}")
print(f"Job Status: {job.status}")
print(f"Creation Time: {job.creation_timestamp}")
print(f"\nJob details: {job}")


Job ID: ea017abeb65c
Job Status: Created
Creation Time: 2026-01-19 13:09:11+00:00

Job details: TrainJob(name='ea017abeb65c', runtime=Runtime(name='torch-distributed', trainer=RuntimeTrainer(trainer_type=<TrainerType.CUSTOM_TRAINER: 'CustomTrainer'>, framework='torch', image='pytorch/pytorch:2.7.1-cuda12.8-cudnn9-runtime', num_nodes=1, device='Unknown', device_count='Unknown'), pretrained_model=None), steps=[Step(name='node-0', status='Pending', pod_name='ea017abeb65c-node-0-0-fkz8b', device='cpu', device_count='1')], num_nodes=1, creation_timestamp=datetime.datetime(2026, 1, 19, 13, 9, 11, tzinfo=TzInfo(0)), status='Created')


In [9]:
print("Waiting for job logs...")
wait_count = 0

while True:
    initial_logs = list(trainer.get_job_logs(job_id, follow=False))
    if initial_logs:
        print(f"Logs received after {wait_count} seconds:")
        for log in initial_logs:
            print(f"  {log}")
        break
    
    wait_count += 1
    print(f"[{datetime.now().strftime('%H:%M:%S')}] Waiting... ({wait_count}s)")
    time.sleep(1)


Waiting for job logs...
[13:09:13] Waiting... (1s)
[13:09:14] Waiting... (2s)
[13:09:15] Waiting... (3s)
[13:09:16] Waiting... (4s)
Logs received after 4 seconds:
  [Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
  PyTorch Distributed Environment
  Using device: cpu
  WORLD_SIZE: 1
  RANK: 0
  LOCAL_RANK: 0


In [7]:
for logline in trainer.get_job_logs(job_id, follow=True):
    print(logline)

[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
PyTorch Distributed Environment
Using device: cpu
WORLD_SIZE: 1
RANK: 0
LOCAL_RANK: 0
